# UTS Data Mining II — LENGKAP (TEXT IMAGE AUDIO)

Versi gabungan dari tiga notebook. Buka bagian yang diperlukan saat UTS. Tiap bagian bisa dijalankan dari atas; notebook gabungan memakai variabel berbeda untuk setiap jenis data.

**Petunjuk:** Buka di Google Colab → `Runtime` → `Run all`, atau jalankan dari atas secara berurutan. Setiap sel **kode** menghasilkan output. Semua angka hasil yang dibahas pada catatan setelah sel adalah panduan interpretasi, **bukan hasil yang diasumsikan sudah diperoleh**.

## Panduan awal — membuka berbagai file

| Jenis file | Perintah paling singkat | Keterangan |
|---|---|---|
| CSV | `pd.read_csv('file.csv')` | Tabel dengan pemisah koma |
| TSV | `pd.read_csv('file.tsv', sep='\t')` | Tabel dengan pemisah tab |
| Excel | `pd.read_excel('file.xlsx')` | Mungkin perlu `openpyxl` |
| JSON | `pd.read_json('file.json')` | Data JSON yang cocok untuk tabel |
| TXT | `open('file.txt', encoding='utf-8').read()` | Teks bebas; tidak harus melalui pandas |
| Gambar | `PIL.Image.open('foto.jpg')` atau `cv2.imread(...)` | **Bukan** dibuka langsung dengan pandas |
| Audio | `librosa.load('audio.wav')` atau `sf.read(...)` | **Bukan** dibuka langsung dengan pandas |

Pandas terutama dipakai untuk data berbentuk tabel dan untuk menampung fitur yang sudah diekstraksi dari teks/gambar/audio. Dataset latihan di notebook ini diakses langsung tanpa perlu upload manual (koneksi internet diperlukan untuk unduhan pertama).

## Contoh tabel pandas

In [ ]:
import pandas as pd
from IPython.display import display

# Contoh objek DataFrame; .head() untuk melihat baris pertama.
contoh = pd.DataFrame({'jenis': ['Text', 'Image', 'Audio'],
                       'fitur': ['TF-IDF', 'Pixel', 'MFCC']})
display(contoh)
print('Ukuran tabel:', contoh.shape)

**Interpretasi:** Output menunjukkan tiga jenis data beserta contoh fitur numeriknya. Untuk pemeriksaan tabel gunakan `.head()`, `.shape`, `.info()`, `.isna().sum()`.

**Keyword:** `pandas read_csv sep tab`, `pandas head shape info`, `read_excel google colab`.

# BAGIAN A — TEXT

## 1. Install library tambahan

In [ ]:
%pip -q install Sastrawi textblob wordcloud
print('Selesai menginstal Sastrawi, TextBlob, dan WordCloud.')

**Interpretasi:** `pip` memasang paket yang belum tersedia. `Sastrawi` adalah paket Python untuk bahasa Indonesia; `TextBlob` dipakai sebagai alat NLP yang utamanya berorientasi bahasa Inggris. **Keyword:** `pip install Sastrawi Colab`, `TextBlob install Python`, `wordcloud package`.

## 2. Import library

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from textblob import TextBlob
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score, davies_bouldin_score
from wordcloud import WordCloud
import nltk

# NLTK adalah library utama untuk sumber tokenizer yang digunakan TextBlob.
print('Paket NLTK punkt_tab berhasil diunduh:', nltk.download('punkt_tab', quiet=True))
print('Import selesai. Import from berarti mengambil class tertentu dari suatu library.')

**Interpretasi:** `sklearn` = library utama scikit-learn, sedangkan `KMeans`, `TfidfVectorizer`, dan `silhouette_score` adalah class/fungsi di submodulnya. `TextBlob` berasal dari `textblob`; `StemmerFactory` berasal dari `Sastrawi`. **Keyword:** `from sklearn.cluster import KMeans`, `TextBlob MissingCorpusError punkt_tab`, `Sastrawi StemmerFactory import`.

## 3. Ambil dataset ulasan Indonesia dari URL

In [ ]:
# URL IndoNLU / SmSA: file TSV tanpa header.
url = 'https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv'
df_asli = pd.read_csv(url, sep='\t', header=None, names=['teks', 'sentimen'])
df_asli = df_asli.dropna(subset=['teks']).reset_index(drop=True)
print('Ukuran dataset asal:', df_asli.shape)
display(df_asli.head(5))

**Interpretasi:** Setiap **baris** merupakan satu dokumen/ulasan. Dataset berlabel tetapi K-Means tetap **unsupervised** karena label tidak dipakai sebagai fitur. Jika URL gagal, periksa koneksi internet Colab. **Keyword:** `pandas read_csv url tsv header None names`, `IndoNLU SmSA Github train_preprocess.tsv`.

## 4. Sampling 120 data

In [ ]:
# Ubah angka 120 menjadi 100, 300, 500, atau 1000 sesuai kebutuhan.
n_sampel = 120
df = df_asli.sample(n=min(n_sampel, len(df_asli)), random_state=42).reset_index(drop=True)
print('Setelah sampling:', df.shape)
print('Distribusi label (hanya informasi):')
print(df['sentimen'].value_counts())
display(df[['teks', 'sentimen']].head(5))

**Interpretasi:** `.sample()` memilih **baris/dokumen**, bukan kata. `random_state=42` membuat pilihan data konsisten. **Keyword:** `pandas sample n random_state`, `sample 100 rows pandas`, `pandas value_counts`.

## 5. Case folding dan cleaning sederhana

In [ ]:
# Huruf kecil, hapus simbol/angka, lalu rapikan spasi.
df['bersih'] = (df['teks'].astype(str).str.lower()
                .str.replace(r'[^a-z\s]', ' ', regex=True)
                .str.replace(r'\s+', ' ', regex=True).str.strip())
display(df[['teks', 'bersih']].head(4))
print('Jumlah dokumen kosong:', (df['bersih'] == '').sum())

**Interpretasi:** **Case folding** menyamakan huruf kapital/kecil; **cleaning** membuang karakter lain. Pola `[a-z]` cocok untuk contoh teks Latin ini, namun angka, tanda baca, dan beberapa informasi dapat hilang. **Keyword:** `python regex remove punctuation`, `pandas str replace regex lowercase`, `text preprocessing case folding`.

## 6. Contoh perbedaan Sastrawi vs TextBlob

In [ ]:
stemmer = StemmerFactory().create_stemmer()
stopword_id = set(StopWordRemoverFactory().get_stop_words())

contoh_id = 'Mahasiswa mempelajari pembelajaran dan menjalankan penelitian.'
contoh_en = 'Children are running with their friends.'
print('Asli Indonesia :', contoh_id)
print('Sastrawi stem  :', stemmer.stem(contoh_id.lower()))
print('TextBlob token :', list(TextBlob(contoh_en).words))
print('Stopword Sastrawi (contoh):', sorted(list(stopword_id))[:15])

**Interpretasi:** **Sastrawi** melakukan stemming bahasa Indonesia (`mempelajari → ajar` biasanya), sedangkan **TextBlob** menyediakan tokenisasi/fitur NLP dan umumnya lebih sesuai untuk **bahasa Inggris**. Dalam notebook ini, TextBlob digunakan untuk **tokenisasi teks Indonesia**, bukan lemmatization Indonesia (tidak tersedia secara andal). **Keyword:** `Sastrawi stem stemming Indonesian`, `TextBlob words tokenize`, `difference stemming lemmatization TextBlob`.

## 7. Pipeline A — preprocessing Sastrawi

In [ ]:
# Gunakan stemmer Indonesia dan buang stopword.
df['teks_sastrawi'] = [
    ' '.join(kata for kata in stemmer.stem(teks).split()
             if kata not in stopword_id and len(kata) > 2)
    for teks in df['bersih']
]
display(df[['bersih', 'teks_sastrawi']].head(5))
print('Contoh jumlah token:', len(df.loc[0, 'teks_sastrawi'].split()))

**Interpretasi:** Kata berimbuhan disederhanakan ke **kata dasar**, kemudian stopword dibuang. Pada data lebih besar proses stemming lebih lambat; bila terburu-buru kurangi `n_sampel` pada tahap 4. **Keyword:** `Sastrawi stemmer.stem example`, `Sastrawi stopword remover factory`, `python list comprehension text preprocessing`.

## 8. WordCloud dan barplot kata terbanyak

In [ ]:
frekuensi = Counter(' '.join(df['teks_sastrawi']).split())
teratas = pd.DataFrame(frekuensi.most_common(12), columns=['kata', 'frekuensi'])
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
wc = WordCloud(width=700, height=350, background_color='white').generate_from_frequencies(frekuensi)
ax[0].imshow(wc)
ax[0].axis('off')
ax[0].set_title('WordCloud - Sastrawi')
ax[1].barh(teratas['kata'][::-1], teratas['frekuensi'][::-1])
ax[1].set_title('12 kata paling sering')
ax[1].set_xlabel('Frekuensi')
plt.tight_layout()
plt.show()
display(teratas)

**Interpretasi:** Kata terbesar pada WordCloud atau barplot = kata yang paling sering muncul **setelah preprocessing**, bukan otomatis kata yang paling penting secara semantik. **Keyword:** `WordCloud generate_from_frequencies Counter`, `python most common words barplot`, `matplotlib barh`.

## 9. TF-IDF dan K-Means pipeline Sastrawi

In [ ]:
# X berupa matriks (jumlah dokumen x jumlah kata/fitur).
vec_s = TfidfVectorizer(min_df=2, max_features=1000)
X_s = vec_s.fit_transform(df['teks_sastrawi'])
print('Shape TF-IDF Sastrawi:', X_s.shape)

km_s = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_sastrawi'] = km_s.fit_predict(X_s)
print('Banyak anggota cluster:')
print(df['cluster_sastrawi'].value_counts().sort_index())

kata_s = vec_s.get_feature_names_out()
for i in range(3):
    idx = km_s.cluster_centers_[i].argsort()[-8:][::-1]
    print('Cluster', i, ':', ', '.join(kata_s[idx]))

**Interpretasi:** TF-IDF mengubah dokumen menjadi **vektor numerik**. `KMeans(n_clusters=3)` membentuk tiga cluster **tanpa memakai label sentimen**. Periksa kata-kata dengan bobot centroid tertinggi untuk menafsirkan topik cluster. Nomor cluster tidak mempunyai arti urutan. **Keyword:** `TfidfVectorizer fit_transform shape`, `KMeans fit_predict sparse tfidf`, `top terms cluster center kmeans`.

## 10. Pipeline B — TextBlob tokenizer pada teks yang sama

In [ ]:
# TextBlob digunakan untuk tokenisasi, bukan stemming/lemma Indonesia.
df['teks_textblob'] = [
    ' '.join(str(kata).lower() for kata in TextBlob(teks).words
             if str(kata).lower() not in stopword_id and len(str(kata)) > 2)
    for teks in df['bersih']
]
display(df[['teks_sastrawi', 'teks_textblob']].head(6))
print('Jumlah dokumen:', len(df))

**Interpretasi:** Hasil TextBlob mempertahankan kata berimbuhan karena contoh ini hanya **tokenization + stopword removal**, sedangkan pipeline Sastrawi menggunakan **stemming**. Jadi hasilnya membandingkan dua **pipeline preprocessing**, bukan kinerja dua stemmer bahasa Indonesia. **Keyword:** `TextBlob Words words tokenization`, `TextBlob bahasa Indonesia stemming`, `TextBlob tokenizer nltk punkt_tab`.

## 11. TF-IDF dan K-Means pipeline TextBlob

In [ ]:
vec_b = TfidfVectorizer(min_df=2, max_features=1000)
X_b = vec_b.fit_transform(df['teks_textblob'])
print('Shape TF-IDF TextBlob:', X_b.shape)

km_b = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_textblob'] = km_b.fit_predict(X_b)
print('Banyak anggota cluster:')
print(df['cluster_textblob'].value_counts().sort_index())

kata_b = vec_b.get_feature_names_out()
for i in range(3):
    idx = km_b.cluster_centers_[i].argsort()[-8:][::-1]
    print('Cluster', i, ':', ', '.join(kata_b[idx]))

**Interpretasi:** Tahapan **TF-IDF → K-Means** sama seperti pada Sastrawi, tetapi representasi kata yang masuk berbeda. Ukuran TF-IDF bisa berbeda karena kosakata hasil cleaning berbeda. **Keyword:** `TextBlob sklearn TF-IDF KMeans`, `how to cluster Indonesian text with sklearn`.

## 12. Evaluasi kedua hasil clustering

In [ ]:
perbandingan = pd.DataFrame({
    'pipeline': ['Sastrawi', 'TextBlob tokenizer'],
    'silhouette': [silhouette_score(X_s, df['cluster_sastrawi']),
                   silhouette_score(X_b, df['cluster_textblob'])],
    'davies_bouldin': [davies_bouldin_score(X_s.toarray(), df['cluster_sastrawi']),
                       davies_bouldin_score(X_b.toarray(), df['cluster_textblob'])]
})
display(perbandingan)
print('Distribusi label referensi vs cluster Sastrawi:')
display(pd.crosstab(df['sentimen'], df['cluster_sastrawi']))

**Interpretasi:** **Silhouette** lebih tinggi (mendekati 1) biasanya lebih baik; **Davies–Bouldin** lebih rendah biasanya lebih baik. Nilai kedua pipeline dipengaruhi perbedaan representasi fitur. Crosstab di sini **bukan** akurasi klasifikasi; label hanya informasi evaluasi eksternal. **Keyword:** `silhouette_score TFIDF KMeans`, `davies_bouldin_score sparse toarray`, `cluster crosstab true label`.

## 13. Visualisasi cluster 2D (TruncatedSVD)

In [ ]:
pos_s = TruncatedSVD(n_components=2, random_state=42).fit_transform(X_s)
pos_b = TruncatedSVD(n_components=2, random_state=42).fit_transform(X_b)
print('Shape SVD Sastrawi:', pos_s.shape)
print('Shape SVD TextBlob:', pos_b.shape)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(pos_s[:, 0], pos_s[:, 1], c=df['cluster_sastrawi'], cmap='viridis', s=22)
ax[0].set_title('Sastrawi + K-Means')
ax[1].scatter(pos_b[:, 0], pos_b[:, 1], c=df['cluster_textblob'], cmap='viridis', s=22)
ax[1].set_title('TextBlob token + K-Means')
for a in ax:
    a.set_xlabel('Komponen 1')
    a.set_ylabel('Komponen 2')
plt.tight_layout()
plt.show()

**Interpretasi:** SVD mereduksi dimensi agar cluster bisa **ditampilkan** pada bidang 2D; K-Means sebelumnya berjalan di ruang TF-IDF aslinya. Bila titik terlihat tumpang tindih, belum tentu seluruh cluster gagal karena ada informasi yang hilang akibat proyeksi. **Keyword:** `TruncatedSVD tfidf 2D visualization`, `matplotlib scatter KMeans text`, `dimensionality reduction text clustering`.

## 14. Lihat contoh dokumen dari tiap cluster

In [ ]:
for c in sorted(df['cluster_sastrawi'].unique()):
    print('\nCLUSTER SASTRAWI', c)
    for kalimat in df.loc[df['cluster_sastrawi'] == c, 'teks'].head(2):
        print('-', kalimat[:180])
print('\nSelesai praktikum teks.')

**Interpretasi:** Cluster wajib ditafsirkan berdasarkan **contoh teks nyata** dan kata dominan, bukan berdasarkan nomor 0/1/2. Alur ringkas: **ambil data → sampling → cleaning → preprocessing → frekuensi → TF-IDF → K-Means → evaluasi → interpretasi**. **Keyword:** `how to interpret text clusters examples`, `extract rows by kmeans cluster pandas`.

# BAGIAN B — IMAGE

## 1. Import library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from skimage import data
print('NumPy = array, matplotlib = plot, sklearn = data + clustering, skimage = contoh foto.')

**Interpretasi:** `from sklearn.datasets import load_digits` mengambil fungsi tertentu dari library utama **scikit-learn**. `from skimage import data` adalah kumpulan contoh gambar. **Keyword:** `sklearn datasets load_digits`, `skimage data astronaut`, `python from import explanation`.

## 2. Dataset gambar angka dan metadata

In [ ]:
digit = load_digits()
gambar = digit.images
print('Total data dan resolusi per gambar:', gambar.shape)
print('Tipe array:', gambar.dtype)
print('Rentang intensitas pixel:', gambar.min(), 'sampai', gambar.max())
print('Contoh label digit (untuk referensi):', digit.target[:12])
plt.figure(figsize=(3, 3))
plt.imshow(gambar[0], cmap='gray')
plt.title('Contoh digit 8 x 8 (grayscale)')
plt.axis('off')
plt.show()

**Interpretasi:** Shape `(jumlah gambar, tinggi, lebar)`; setiap gambar berukuran **8 × 8 pixel** dan satu kanal grayscale. Intensitas pada `load_digits` adalah **0–16**, bukan rentang 8-bit 0–255. **Keyword:** `numpy image shape dtype min max`, `grayscale image 8x8 sklearn digits`, `image resolution pixel channel`.

## 3. Sampling 500 gambar

In [ ]:
n_sampel = 500  # Bisa diganti 100, 300, 1000 (asal <= jumlah gambar).
rng = np.random.default_rng(42)
indeks = rng.choice(len(gambar), size=min(n_sampel, len(gambar)), replace=False)
gambar_sample = gambar[indeks]
label_asli = digit.target[indeks]  # Hanya untuk pemeriksaan; tidak dipakai K-Means.
print('Shape sebelum sampling:', gambar.shape)
print('Shape setelah sampling:', gambar_sample.shape)
print('Shape label referensi:', label_asli.shape)

**Interpretasi:** Sampling diambil per **gambar**, tanpa pengembalian (`replace=False`). Setiap data yang dipilih masih berbentuk matriks **8×8**. **Keyword:** `numpy random choice replace False`, `random sampling images numpy`, `image array shape after sampling`.

## 4. Flatten: gambar 2D menjadi vektor 1D

In [ ]:
X_digit = gambar_sample.reshape(len(gambar_sample), -1)
print('Sebelum flatten:', gambar_sample.shape)
print('Sesudah flatten:', X_digit.shape)
print('Contoh fitur gambar pertama (64 pixel):')
print(X_digit[0])

**Interpretasi:** Format masukan K-Means = `(n_samples, n_features)`. Bentuk `(500, 8, 8)` menjadi `(500, 64)` karena **8×8 = 64 pixel per gambar**. `-1` membuat NumPy menghitung otomatis panjang vektornya. **Keyword:** `numpy reshape n_samples -1 image flatten`, `ValueError Found array with dim 3 KMeans`, `reshape 2D image for machine learning`.

## 5. Clustering K-Means gambar angka

In [ ]:
km_digit = KMeans(n_clusters=10, random_state=42, n_init=10)
cluster_digit = km_digit.fit_predict(X_digit)
print('Shape data yang masuk K-Means:', X_digit.shape)
print('Shape hasil label cluster:', cluster_digit.shape)
print('Jumlah data per cluster:')
print(np.bincount(cluster_digit, minlength=10))
print('Silhouette:', round(silhouette_score(X_digit, cluster_digit), 3))

**Interpretasi:** `n_clusters=10` dipilih sebagai latihan karena ada 10 jenis digit (0–9), **bukan** berarti K-Means tahu identitas digit tersebut. Silhouette mengukur pemisahan cluster; mendekati 1 lebih baik. **Keyword:** `KMeans images digits dataset`, `sklearn silhouette_score image features`, `kmeans cluster sizes np bincount`.

## 6. Visualisasi sampel digit per cluster

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(11, 5))
for c in range(10):
    idx = np.flatnonzero(cluster_digit == c)[0]
    ax.flat[c].imshow(gambar_sample[idx], cmap='gray')
    ax.flat[c].set_title(f'Cluster {c}')
    ax.flat[c].axis('off')
plt.tight_layout()
plt.show()
print('Masing-masing panel menunjukkan satu contoh data dari cluster terkait.')

**Interpretasi:** Tampilan hanya mengambil **satu contoh anggota tiap cluster**, bukan pusat cluster. Bandingkan bentuk karakter antarcluster; satu digit asli bisa tersebar pada beberapa cluster. **Keyword:** `matplotlib imshow subplots kmeans image`, `numpy flatnonzero cluster indices`.

## 7. Visualisasi cluster melalui PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_digit)
print('Shape sebelum PCA:', X_digit.shape)
print('Shape setelah PCA:', X_pca.shape)
plt.figure(figsize=(7, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_digit, cmap='tab10', s=16)
plt.title('Clustering gambar digit — PCA 2D')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(label='Nomor cluster')
plt.show()

**Interpretasi:** PCA mengubah **64 fitur pixel menjadi 2 komponen** untuk plotting. Clustering tetap menggunakan data 64 fitur sebelumnya. PCA tidak sama dengan flatten. **Keyword:** `PCA(n_components=2) image clustering`, `PCA plot KMeans digits`, `difference flatten and PCA`.

## 8. Contoh gambar RGB dan informasi resolusi

In [ ]:
rgb = data.astronaut()
print('Shape awal gambar RGB:', rgb.shape)
print('Resolusi (tinggi x lebar):', rgb.shape[0], 'x', rgb.shape[1])
print('Jumlah channel:', rgb.shape[2])
print('Tipe pixel:', rgb.dtype)
print('Rentang nilai:', rgb.min(), rgb.max())
plt.figure(figsize=(5, 4))
plt.imshow(rgb)
plt.title('Foto RGB - skimage.data.astronaut()')
plt.axis('off')
plt.show()

**Interpretasi:** Gambar warna berupa `(tinggi, lebar, 3)` dengan kanal **R, G, B**. Nilai pixel `uint8` lazimnya 0–255. RGB dari scikit-image berbeda dengan default BGR saat dibaca dengan OpenCV (`cv2.imread`). **Keyword:** `skimage.data.astronaut RGB shape`, `OpenCV BGR vs RGB`, `numpy image metadata channels`.

## 9. Flatten semua pixel dan sampling 1000 pixel

In [ ]:
pixel_flat = rgb.reshape(-1, 3)
print('Shape sebelum flatten:', rgb.shape)
print('Shape setelah flatten:', pixel_flat.shape)

indeks_pixel = np.random.default_rng(42).choice(len(pixel_flat), size=1000, replace=False)
X_pixel = pixel_flat[indeks_pixel]
print('Shape setelah sampling 1000 pixel:', X_pixel.shape)
print('Contoh 5 pixel RGB:')
print(X_pixel[:5])

**Interpretasi:** Kini **unit data = satu pixel**, bukan satu foto. `(H,W,3)` diubah menjadi `(H×W,3)`; tiga fiturnya adalah R,G,B. Jangan memakai `reshape(n_gambar,-1)` untuk kasus segmentasi warna ini karena unit sampelnya berbeda. **Keyword:** `RGB pixel flatten reshape -1 3`, `KMeans color quantization sampling pixels`, `numpy sample rows rgb`.

## 10. K-Means segmentasi warna dan reshape kembali

In [ ]:
km_warna = KMeans(n_clusters=4, random_state=42, n_init=10)
km_warna.fit(X_pixel)
label_semua_pixel = km_warna.predict(pixel_flat)
warna_segmentasi = km_warna.cluster_centers_[label_semua_pixel].astype('uint8')
gambar_segmentasi = warna_segmentasi.reshape(rgb.shape)
print('Shape prediksi semua pixel:', label_semua_pixel.shape)
print('Shape warna dari cluster:', warna_segmentasi.shape)
print('Shape sesudah reshape kembali:', gambar_segmentasi.shape)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(rgb)
ax[0].set_title('Asli')
ax[1].imshow(gambar_segmentasi)
ax[1].set_title('Hasil K-Means 4 warna')
for a in ax: a.axis('off')
plt.tight_layout()
plt.show()

**Interpretasi:** Cluster warna diwakili **centroid RGB**; hasil `predict` dipetakan lagi menjadi citra berdimensi asli. Jadi **clustering pixel** di sini berfungsi untuk segmentasi/penyederhanaan warna, bukan clustering antargambar. **Keyword:** `KMeans image segmentation color quantization`, `KMeans predict all pixels reshape`, `kmeans centroid RGB image`.

## 11. Ringkasan 2 tugas image

In [ ]:
print('A — Clustering antar gambar:')
print('    gambar_sample:', gambar_sample.shape, '-> X_digit:', X_digit.shape)
print('B — Segmentasi warna pixel:')
print('    rgb:', rgb.shape, '-> pixel_flat:', pixel_flat.shape, '-> hasil:', gambar_segmentasi.shape)
print('Selesai praktikum image.')

**Interpretasi:** Jika soal menyebut **kelompok foto**: buat fitur per foto `(n_foto, n_fitur)`. Jika soal menyebut **segmentasi warna dalam satu foto**: buat fitur per pixel `(n_pixel, 3)`. **Keyword:** `image clustering vs pixel clustering`, `how to choose samples and features image kmeans`.

# BAGIAN C — AUDIO

## 1. Import library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from IPython.display import display, Audio
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
print('librosa = pemrosesan suara, soundfile = info file/WAV, sklearn = clustering.')
print('Versi librosa:', librosa.__version__)

**Interpretasi:** `librosa` adalah library utama; `librosa.display` submodul untuk visualisasi. `sf` adalah alias dari library **soundfile** yang membaca metadata audio. **Keyword:** `librosa display specshow`, `soundfile info channels samplerate subtype`, `from sklearn.preprocessing import StandardScaler`.

## 2. Unduh contoh audio dari katalog librosa

In [ ]:
# Unduhan pertama butuh internet; berikutnya tersimpan di cache Colab.
nama_audio = ['trumpet', 'brahms', 'nutcracker']
berkas_audio = [librosa.ex(nama) for nama in nama_audio]
print('Daftar audio bawaan:')
for nama, path in zip(nama_audio, berkas_audio):
    print(nama, '->', path)

**Interpretasi:** `librosa.ex("trumpet")` mengembalikan **path** audio contoh, bukan langsung array sinyal. Tiga sumber digunakan supaya potongan audio punya variasi. **Keyword:** `librosa ex trumpet brahms nutcracker`, `librosa util list_examples`, `librosa example recordings`.

## 3. Muat waveform dan dengarkan

In [ ]:
# sr=22050 artinya audio di-resample ke 22.050 sampel/detik dan mono (default).
y, sr = librosa.load(berkas_audio[0], sr=22050)
print('Shape waveform mono:', y.shape)
print('Sample rate:', sr, 'Hz')
print('Durasi:', round(len(y) / sr, 2), 'detik')
print('Amplitudo min/max:', round(float(y.min()), 3), round(float(y.max()), 3))
display(Audio(y, rate=sr))

**Interpretasi:** `y` adalah sinyal **1D** karena dimuat mono. `sr` = sampling rate. `Audio(y,rate=sr)` dapat diputar langsung di Colab. `librosa.load` default melakukan mono conversion; ini belum tentu sama dengan jumlah channel pada file aslinya. **Keyword:** `librosa.load sr mono audio shape`, `IPython display Audio`, `audio sampling rate duration`.

## 4. Metadata channel, bit depth, bitrate WAV

In [ ]:
# Menyimpan waveform hasil pemuatan menjadi WAV PCM 16-bit, untuk contoh metadata pasti.
sf.write('contoh_audio_pcm16.wav', y, sr, subtype='PCM_16')
info = sf.info('contoh_audio_pcm16.wav')
bit_depth = 16  # Sesuai subtype yang ditentukan saat menulis WAV.
bitrate_pcm = info.samplerate * info.channels * bit_depth
print('Format / subtype:', info.format, '/', info.subtype)
print('Channel:', info.channels, '(1=mono; 2=stereo)')
print('Sample rate:', info.samplerate, 'Hz')
print('Frames/samples per channel:', info.frames)
print('Durasi:', round(info.duration, 2), 'detik')
print('Bit depth:', bit_depth, 'bit/sample')
print('Bitrate PCM (perkiraan tanpa header):', bitrate_pcm, 'bit/s =', round(bitrate_pcm/1000, 1), 'kbps')

**Interpretasi:** **Sample rate** = sampel per detik; **bit depth** = bit tiap sampel; **channel** = mono/stereo; **bitrate PCM** = `sample_rate × bit_depth × channel`. Ini berlaku untuk **WAV PCM yang baru dibuat**, bukan otomatis bitrate pada file MP3/OGG asli (yang terkompresi). **Keyword:** `soundfile info subtype PCM_16`, `audio bitrate sample rate bit depth channels formula`, `mono stereo audio metadata`.

## 5. Visualisasi waveform amplitudo terhadap waktu

In [ ]:
plt.figure(figsize=(12, 3))
librosa.display.waveshow(y, sr=sr)
plt.title('Waveform — amplitudo terhadap waktu')
plt.xlabel('Waktu (detik)')
plt.ylabel('Amplitudo (skala sinyal)')
plt.tight_layout()
plt.show()

**Interpretasi:** Waveform memperlihatkan variasi **amplitudo terhadap waktu**. Nilai amplitudo normalisasi bukan langsung nilai **dB**. **Keyword:** `librosa.display.waveshow waveform`, `audio waveform amplitude versus time`.

## 6. Spectrogram magnitudo/amplitudo

In [ ]:
# STFT mengubah sinyal waktu menjadi waktu × frekuensi.
S = np.abs(librosa.stft(y, n_fft=1024, hop_length=256))
print('Shape spektrogram magnitudo:', S.shape)
plt.figure(figsize=(11, 4))
librosa.display.specshow(S, sr=sr, hop_length=256, x_axis='time', y_axis='linear')
plt.colorbar(label='Magnitudo STFT (skala linear)')
plt.title('Spectrogram — magnitudo / amplitudo')
plt.tight_layout()
plt.show()

**Interpretasi:** Sumbu x = waktu, y = frekuensi dalam Hz, warna = **magnitudo komponen frekuensi**. STFT adalah *Short-Time Fourier Transform*. Frekuensi maksimum ≈ `sr/2` (Nyquist). **Keyword:** `librosa stft abs amplitude spectrogram`, `librosa.display.specshow x_axis time y_axis linear`, `n_fft hop_length explanation`.

## 7. Spectrogram skala desibel (dB)

In [ ]:
S_db = librosa.amplitude_to_db(S, ref=np.max)
print('Shape sebelum (magnitudo):', S.shape)
print('Shape sesudah (dB):', S_db.shape)
print('Rentang dB:', round(float(S_db.min()), 1), 'sampai', round(float(S_db.max()), 1))
plt.figure(figsize=(11, 4))
librosa.display.specshow(S_db, sr=sr, hop_length=256, x_axis='time', y_axis='linear')
plt.colorbar(label='dB relatif terhadap magnitudo maksimum')
plt.title('Spectrogram — skala dB')
plt.tight_layout()
plt.show()

**Interpretasi:** dB mengubah **skala tampilan** magnitudo menjadi logaritmik. `ref=np.max` membuat puncak relatif = **0 dB** dan komponen lebih lemah bernilai negatif. Ukuran matriks tidak berubah. **Keyword:** `librosa.amplitude_to_db ref np max`, `difference amplitude spectrogram and dB spectrogram`, `STFT spectrogram log scale`.

## 8. Ekstraksi MFCC dari potongan tiap sumber audio

In [ ]:
# Unit data clustering = potongan/segment berdurasi 1 detik; langkah antarsegmen 0,5 detik.
baris = []
for nama, path in zip(nama_audio, berkas_audio):
    sinyal, fs = librosa.load(path, sr=22050, duration=25)
    panjang = fs               # 1 detik
    langkah = fs // 2          # Geser 0,5 detik
    for awal in range(0, len(sinyal) - panjang + 1, langkah):
        potongan = sinyal[awal:awal + panjang]
        mfcc = librosa.feature.mfcc(y=potongan, sr=fs, n_mfcc=13)
        baris.append([nama, awal / fs] + mfcc.mean(axis=1).tolist())

kolom = ['sumber', 'waktu_mulai'] + [f'mfcc_{k}' for k in range(1, 14)]
df_audio_asli = pd.DataFrame(baris, columns=kolom)
print('Shape seluruh potongan audio:', df_audio_asli.shape)
display(df_audio_asli.head())

**Interpretasi:** **MFCC** adalah fitur numerik karakteristik spektral suara. Setiap potongan menghasilkan 13 fitur (rata-rata koefisien sepanjang frame waktu), ditambah `sumber` dan `waktu_mulai` sebagai **metadata**, bukan fitur model. Satu data clustering = satu **potongan audio**, bukan satu sampel waveform. **Keyword:** `librosa feature mfcc y sr n_mfcc`, `audio split into 1 second segments Python`, `MFCC mean axis 1`, `audio clustering features`.

## 9. Sampling 100 segmen dan standardisasi fitur

In [ ]:
df_audio = df_audio_asli.sample(n=min(100, len(df_audio_asli)), random_state=42).reset_index(drop=True)
X_audio = df_audio[[f'mfcc_{k}' for k in range(1, 14)]].to_numpy()
scaler = StandardScaler()
X_audio_scaled = scaler.fit_transform(X_audio)
print('Jumlah data setelah sampling:', len(df_audio))
print('Shape fitur sebelum standardisasi:', X_audio.shape)
print('Shape fitur setelah standardisasi:', X_audio_scaled.shape)
print('Distribusi sumber:')
print(df_audio['sumber'].value_counts())

**Interpretasi:** `.sample()` memilih maksimal 100 **segmen**. `StandardScaler` menyetarakan skala koefisien sebelum K-Means. Bentuk `(n_segmen,13)` tidak berubah saat standardisasi. **Keyword:** `sample dataframe 100 rows`, `StandardScaler fit_transform MFCC`, `why standardize before KMeans`.

## 10. Clustering K-Means dan evaluasi

In [ ]:
km_audio = KMeans(n_clusters=3, random_state=42, n_init=10)
df_audio['cluster'] = km_audio.fit_predict(X_audio_scaled)
print('Shape fitur input:', X_audio_scaled.shape)
print('Shape cluster output:', df_audio['cluster'].shape)
print('Jumlah tiap cluster:')
print(df_audio['cluster'].value_counts().sort_index())
print('Silhouette:', round(silhouette_score(X_audio_scaled, df_audio['cluster']), 3))
print('Crosstab sumber vs cluster:')
display(pd.crosstab(df_audio['sumber'], df_audio['cluster']))

**Interpretasi:** K-Means mengelompokkan **karakteristik MFCC**. Distribusi cluster antar sumber bisa membantu interpretasi, tetapi tidak berarti kita melakukan klasifikasi instrumen atau genre secara akurat. **Keyword:** `KMeans clustering MFCC audio`, `silhouette_score sklearn audio`, `pandas crosstab source cluster`.

## 11. Visualisasi cluster menggunakan PCA 2D

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_audio_2d = pca.fit_transform(X_audio_scaled)
print('Shape MFCC sebelumnya:', X_audio_scaled.shape)
print('Shape sesudah PCA:', X_audio_2d.shape)
plt.figure(figsize=(7, 5))
plt.scatter(X_audio_2d[:, 0], X_audio_2d[:, 1], c=df_audio['cluster'], cmap='viridis', s=24)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means segmen audio (fitur MFCC)')
plt.colorbar(label='Cluster')
plt.tight_layout()
plt.show()

**Interpretasi:** PCA mengubah 13 koefisien MFCC menjadi 2 dimensi **hanya untuk visualisasi**. K-Means berjalan pada 13 fitur terstandarisasi. **Keyword:** `PCA MFCC audio KMeans scatter`, `audio clustering visualize PCA 2 components`.

## 12. Ringkasan metadata dan alur audio

In [ ]:
print('METADATA WAV: channel =', info.channels, ', sample rate =', info.samplerate,
      ', bit depth =', bit_depth, ', bitrate PCM =', bitrate_pcm, 'bit/s')
print('SPECTROGRAM:', S.shape, '-> dB:', S_db.shape)
print('CLUSTERING:', df_audio_asli.shape, '-> sampling:', X_audio.shape,
      '-> scaling:', X_audio_scaled.shape, '-> PCA plot:', X_audio_2d.shape)
print('Selesai praktikum audio.')

**Interpretasi:** Alur utama: **baca file → cek metadata → waveform/spectrogram → potong audio → MFCC → sampling → scaling → K-Means → evaluasi → interpretasi**. Jangan menyamakan **bitrate**, **sample rate**, **bit depth**, **amplitudo**, dan **dB**. **Keyword:** `audio digital signal processing metadata sample rate bit depth`, `MFCC KMeans clustering workflow`.